# r3con on a self-hosted model

The pipeline talks to any OpenAI-compatible endpoint — vLLM, Ollama, TGI, an internal
gateway. Two things change versus a hosted provider: the `model` string carries litellm's
provider prefix, and you pass the `api_base` yourself. The key is read from litellm's own
provider variable, exactly as `OPENAI_API_KEY` would be.

```bash
vllm serve Qwen/Qwen3.5-35B-A3B --port 8555 --api-key your_secret
export HOSTED_VLLM_API_KEY=your_secret
```


In [ ]:
from r3con import r3con

MODEL    = "hosted_vllm/Qwen/Qwen3.5-35B-A3B"   # hosted_vllm/ + the model you served
API_BASE = "http://localhost:8555/v1"           # the key comes from HOSTED_VLLM_API_KEY

QUESTION = ("Which contractor was responsible for the most equipment incidents across our "
            "sites in Q3, and how many? Give the contractor's name, not its code.")

# The correct answer, so you can check the run rather than trust it:
#
#   Halloran Services Ltd, with 11 incidents.
#   CT-118 -> Northgate 5 + Riverside 6 = 11, and the registry names CT-118.
#
# No single memo holds it: the counts name only a code, and the code->name mapping
# lives in a document that has no counts.

docs = r3con.read_documents("memos")
result = r3con.run(QUESTION, docs, model=MODEL, api_base=API_BASE)
print(result)

Halloran Services Ltd was responsible for the most equipment incidents in Q3, with a total of 11 incidents across the Northgate and Riverside sites.


## What the run cost

A thinking model spends most of its tokens before it answers. Every call is already
recorded in the run folder, so the bill is readable after the fact — worth checking
before you point this at a large corpus.


In [ ]:
import json

for stage in ["relevance", "structuring/schema", "structuring/parsing", "reasoning"]:
    t = json.load(open(f"{result.run_dir}/{stage}/result.json"))["totals"]
    print(f"{stage:22} {t['n_steps']:2} calls  {t['tokens']['completion']:6,} completion tokens")

relevance              10 calls  26,926 completion tokens
structuring/schema      1 calls   5,602 completion tokens
structuring/parsing     5 calls  15,989 completion tokens
reasoning               2 calls     770 completion tokens
